# V12C — Rebuild training epoch timestamps

**Purpose.** Reconstruct the absolute start time of every preictal and
interictal epoch used by `ModelReady_GAPAWARE_V4`, without recomputing any
handcrafted, PLI, wPLI, or graph features.

This notebook replays the original gap-aware segmentation rules from the
uploaded segmentation source and creates a small timestamp dataset for the
V12B `GUARD10` and chronological `FORWARD` analyses.

## Required Kaggle inputs

1. The original raw CHB–MIT EDF dataset (patient folders such as `chb01`).
2. The complete `ModelReady_GAPAWARE_V4` dataset used by V12B.

CPU is sufficient. GPU is not required.

## Non-negotiable validity rule

The notebook writes a usable final manifest only when every ModelReady
training epoch is matched exactly once by
`Patient + Class + SeizureID + Epoch`. Any mismatch raises an error.


In [ ]:
# ==========================================================================
# 0. IMPORTS AND LOCKED PARAMETERS FROM THE ORIGINAL SEGMENTATION
# ==========================================================================
import csv
import gc
import json
import os
import re
import shutil
import traceback
from datetime import timedelta
from pathlib import Path

import mne
import numpy as np
import pandas as pd

EXPECTED_FS = 256.0
SAMPLE_PERIOD_SEC = 1.0 / EXPECTED_FS
PREICTAL_SEC = 30 * 60
INTERICTAL_DIST_SEC = 4 * 60 * 60
POSTICTAL_SEC = 60 * 60
CONTINUOUS_TOLERANCE_SEC = SAMPLE_PERIOD_SEC + 1e-9
MAX_SHORT_GAP_SEC = 10.0
EPOCH_LEN_SEC = 10

EXPECTED_PREICTAL_EPOCHS = PREICTAL_SEC // EPOCH_LEN_SEC       # 180
MIN_INTERICTAL_EPOCHS_PER_SEIZURE = EXPECTED_PREICTAL_EPOCHS * 3
MAX_INTERICTAL_EPOCHS_PER_SEIZURE = MIN_INTERICTAL_EPOCHS_PER_SEIZURE  # 540
MIN_ELIGIBLE_SEIZURES_PER_PATIENT = 3
PREICTAL_MAX_LOOKBACK_SEC = 35 * 60
INTERICTAL_ASSIGNMENT_CHUNK_SEC = 30 * 60
BATCH_SIZE = 1800  # retained for compatibility with an unused V3 helper

INPUT_SEARCH_ROOT = Path('/kaggle/input')
MODEL_READY_ROOT_OVERRIDE = None
RAW_CHBMIT_ROOT_OVERRIDE = None
OUTPUT_ROOT = Path('/kaggle/working/MBEC_V12C_TRAINING_TIMESTAMPS')

# True only when intentionally restarting V12C from zero.
RESET_OUTPUT = True

# Set to a list such as ['chb01'] for a quick pilot. None processes every
# subject present in ModelReady_GAPAWARE_V4.
SUBJECTS_OVERRIDE = None

CHANNELS_18 = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ',
]

mne.set_log_level('WARNING')


In [ ]:
# ==========================================================================
# 1. DISCOVER INPUT DATASETS AND THE EXACT V12 COHORT
# ==========================================================================
def discover_model_ready_root():
    candidates = ([Path(MODEL_READY_ROOT_OVERRIDE)]
                  if MODEL_READY_ROOT_OVERRIDE is not None
                  else list(INPUT_SEARCH_ROOT.rglob('ModelReady_GAPAWARE_V4')))
    valid = []
    for root in candidates:
        required = [root / 'Metadata', root / 'Handcrafted',
                    root / 'Graph_PLI', root / 'Graph_wPLI',
                    root / 'feature_set_definitions.json']
        if all(path.exists() for path in required):
            valid.append(root.resolve())
    valid = sorted(set(valid))
    if not valid:
        raise FileNotFoundError(
            'Could not find complete ModelReady_GAPAWARE_V4 under /kaggle/input')
    if len(valid) > 1:
        raise RuntimeError(
            f'Multiple ModelReady roots found: {valid}. '
            'Set MODEL_READY_ROOT_OVERRIDE explicitly.')
    return valid[0]


def discover_chbmit_root():
    if RAW_CHBMIT_ROOT_OVERRIDE is not None:
        candidates = [Path(RAW_CHBMIT_ROOT_OVERRIDE)]
    else:
        candidates = []
        for chb01_dir in INPUT_SEARCH_ROOT.rglob('chb01'):
            if (chb01_dir.is_dir()
                    and any(chb01_dir.glob('*.edf'))
                    and any(chb01_dir.glob('*summary.txt'))):
                candidates.append(chb01_dir.parent)
    candidates = sorted(set(path.resolve() for path in candidates),
                        key=lambda path: (len(path.parts), str(path)))
    if not candidates:
        raise FileNotFoundError(
            'Could not locate raw CHB-MIT EDF data. Attach the raw dataset.')
    if len(candidates) > 1:
        print('[WARN] Multiple raw roots found; using:', candidates[0])
        print('Other candidates:', candidates[1:])
    return candidates[0]


MODEL_ROOT = discover_model_ready_root()
RAW_ROOT = discover_chbmit_root()

subjects = sorted(
    path.name for path in (MODEL_ROOT / 'Metadata').iterdir()
    if path.is_dir() and re.fullmatch(r'chb\d{2}', path.name))
if SUBJECTS_OVERRIDE is not None:
    requested = set(SUBJECTS_OVERRIDE)
    missing = sorted(requested - set(subjects))
    if missing:
        raise ValueError(f'SUBJECTS_OVERRIDE not present in ModelReady: {missing}')
    subjects = [subject for subject in subjects if subject in requested]
if not subjects:
    raise RuntimeError('No V12 subjects found')

if RESET_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
(OUTPUT_ROOT / 'subjects').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)

print('ModelReady root:', MODEL_ROOT)
print('Raw CHB-MIT root:', RAW_ROOT)
print(f'Subjects ({len(subjects)}):', subjects)
print('Output:', OUTPUT_ROOT)


## Original gap-aware timeline and eligibility logic

The following functions are preserved from the original segmentation notebook.


In [ ]:
# 1. CHANNEL SELECTION AND SEIZURE ANNOTATIONS
# ==========================================================================
def select_channels(raw, channel_list=CHANNELS_18):
    """Require all channels and force one invariant channel order.

    A recording with a missing required channel is excluded; it is never allowed
    to produce a smaller matrix with shifted channel indices.
    """
    if 'T8-P8-0' in raw.ch_names:
        raw.rename_channels({'T8-P8-0': 'T8-P8'})
    raw.rename_channels({ch: ch.strip().replace('EEG ', '').upper()
                         for ch in raw.ch_names})

    missing = [ch for ch in channel_list if ch not in raw.ch_names]
    if missing:
        raise ValueError(f'missing required channels: {missing}')

    raw.pick(channel_list)
    raw.reorder_channels(channel_list)
    if raw.ch_names != list(channel_list):
        raise AssertionError('channel order is not invariant')
    return raw


def load_seizures_from_summary(summary_path, edf_filename):
    seizures = []
    if not os.path.exists(summary_path):
        return seizures
    with open(summary_path, 'r', encoding='latin-1') as f:
        content = f.read()
    for file_block in content.split('File Name:'):
        if edf_filename.lower() not in file_block.lower():
            continue
        n_match = re.search(r'Number of Seizures in File:\s*(\d+)', file_block)
        if not n_match or int(n_match.group(1)) == 0:
            return seizures
        onsets = re.findall(r'Seizure\s*(?:\d+)?\s*Start Time:\s*(\d+)\s*seconds', file_block)
        offsets = re.findall(r'Seizure\s*(?:\d+)?\s*End Time:\s*(\d+)\s*seconds', file_block)
        seizures = [(float(on), float(off)) for on, off in zip(onsets, offsets)]
        break
    return seizures



# ==========================================================================


In [ ]:
# 2. BUILD THE RECORDING TIMELINE OF ONE PATIENT
# ==========================================================================
def _discover_edf_files(subject_dir):
    return sorted(f for f in os.listdir(subject_dir) if f.lower().endswith('.edf'))


def _read_file_infos(subject_dir, subject, edf_files):
    """Read every EDF header and attach seizure annotations."""
    summary_path = os.path.join(subject_dir, f'{subject}-summary.txt')
    file_infos = []
    for f in edf_files:
        raw = mne.io.read_raw_edf(os.path.join(subject_dir, f), preload=False, verbose=False)
        try:
            raw = select_channels(raw, CHANNELS_18)
        except ValueError as exc:
            print(f'[EXCLUDE EDF] {f}: {exc}')
            continue
        if not np.isclose(float(raw.info['sfreq']), EXPECTED_FS):
            print(f"[EXCLUDE EDF] {f}: sfreq={raw.info['sfreq']} Hz, expected {EXPECTED_FS} Hz")
            continue
        meas_date = raw.info['meas_date']
        if meas_date is None:
            print(f'[WARN] {f}: no meas_date -> skipping file')
            continue
        duration = raw.times[-1] + 1.0 / raw.info['sfreq']
        file_infos.append({
            'file': f, 'raw': raw, 'start': meas_date, 'duration': duration,
            'end': meas_date + timedelta(seconds=duration),
            'seizures': load_seizures_from_summary(summary_path, f),
        })
    file_infos.sort(key=lambda x: x['start'])
    return file_infos


def _group_into_blocks(file_infos):
    """Keep gaps <=10 s in one logical block; split larger gaps and overlaps."""
    if not file_infos:
        return []
    blocks = [[file_infos[0]]]
    for prev, cur in zip(file_infos, file_infos[1:]):
        gap = (cur['start'] - prev['end']).total_seconds()
        if -CONTINUOUS_TOLERANCE_SEC <= gap <= MAX_SHORT_GAP_SEC:
            if gap > CONTINUOUS_TOLERANCE_SEC:
                print(f'[TOLERATED SHORT GAP] {prev["file"]} -> {cur["file"]}: {gap:.3f} s')
            blocks[-1].append(cur)
        else:
            kind = 'OVERLAP' if gap < 0 else 'GAP'
            print(f'[SPLIT BLOCK: {kind}] {prev["file"]} -> {cur["file"]}: {gap:.3f} s')
            blocks.append([cur])
    return blocks


def _concat_block(block_files):
    """Concatenate EDF samples while retaining MNE boundary annotations."""
    raws, data_cursor = [], 0.0
    for fi in block_files:
        fi['data_start'] = data_cursor
        raw = fi['raw'].copy().load_data()
        raws.append(raw)
        data_cursor += fi['duration']
    raw_concat = raws[0] if len(raws) == 1 else mne.concatenate_raws(raws)
    # BAD/EDGE boundary annotations deliberately remain: reject_by_annotation=True
    # then prevents a 10-s epoch from containing samples from two EDF files.
    raw_concat.notch_filter(freqs=60.0, verbose=False)
    return raw_concat, block_files[0]['start']


def data_sec_to_abs(block, sec):
    """Map compressed concatenated-data seconds to the original EDF clock."""
    eps = SAMPLE_PERIOD_SEC + 1e-9
    for idx, fi in enumerate(block['files']):
        lo, hi = fi['data_start'], fi['data_start'] + fi['duration']
        is_last = idx == len(block['files']) - 1
        # Concatenation boundaries belong to the following EDF, not the
        # preceding EDF's wall-clock end (which may be separated by a gap).
        if lo - eps <= sec < hi or (is_last and sec <= hi + eps):
            within = min(max(sec - lo, 0.0), fi['duration'])
            return fi['start'] + timedelta(seconds=within)
    raise ValueError(f'data second {sec} is outside block')


def abs_to_data_sec(block, when):
    """Map an EDF-clock datetime to compressed concatenated-data seconds."""
    for fi in block['files']:
        if fi['start'] <= when <= fi['end']:
            return fi['data_start'] + (when - fi['start']).total_seconds()
    # A requested interval endpoint may fall inside a tolerated gap. Map it to
    # the adjacent sample boundary; the missing wall-clock time is never invented.
    for prev, cur in zip(block['files'], block['files'][1:]):
        if prev['end'] < when < cur['start']:
            return cur['data_start']
    raise ValueError(f'absolute time {when} is outside recorded EDF data')


def build_timeline(subject_dir, subject):
    edf_files = _discover_edf_files(subject_dir)
    file_infos = _read_file_infos(subject_dir, subject, edf_files)
    blocks = []
    for block_files in _group_into_blocks(file_infos):
        raw_concat, block_start = _concat_block(block_files)
        blocks.append({
            'raw': raw_concat, 'start': block_start,
            'duration': raw_concat.times[-1] + 1.0 / raw_concat.info['sfreq'],
            'end': block_files[-1]['end'], 'files': block_files,
        })
    blocks.sort(key=lambda b: b['start'])
    return blocks



# ==========================================================================


In [ ]:
# 3. COLLECT ALL SEIZURES OF A PATIENT
# ==========================================================================
def collect_seizures(blocks):
    """Return seizures with compressed-data and absolute EDF-clock coordinates."""
    all_seizures = []
    for b_idx, block in enumerate(blocks):
        for finfo in block['files']:
            for onset, offset in finfo['seizures']:
                all_seizures.append({
                    'block_idx': b_idx,
                    'onset_in_block': finfo['data_start'] + onset,
                    'offset_in_block': finfo['data_start'] + offset,
                    'onset_abs': finfo['start'] + timedelta(seconds=onset),
                    'offset_abs': finfo['start'] + timedelta(seconds=offset),
                    'source_file': finfo['file'],
                })
    all_seizures.sort(key=lambda s: s['onset_abs'])
    for i, s in enumerate(all_seizures):
        s['seizure_id'] = i + 1
    return all_seizures



# ==========================================================================


In [ ]:
# 4. GAP-AWARE PREICTAL DEFINITION
# ==========================================================================
def get_previous_seizure(seizure, all_seizures):
    earlier = [s for s in all_seizures if s['onset_abs'] < seizure['onset_abs']]
    return max(earlier, key=lambda s: s['onset_abs']) if earlier else None


def get_preictal_window(seizure, block, all_seizures):
    """Return a candidate window containing up to 35 min before onset.

    The final extractor retains the latest 180 valid 10-s epochs. This permits
    short EDF gaps while still requiring exactly 30 min of available EEG.
    """
    onset = seizure['onset_in_block']
    if onset < PREICTAL_SEC:
        return None
    return max(0.0, onset - PREICTAL_MAX_LOOKBACK_SEC), onset



# ==========================================================================


In [ ]:
# 5. INTERICTAL WINDOWS AND NON-OVERLAPPING LOSO ALLOCATION
# ==========================================================================
def get_interictal_windows(blocks, all_seizures):
    """Return valid interictal windows separately inside each physical EDF file."""
    seizures = sorted(all_seizures, key=lambda s: s['onset_abs'])
    margin = timedelta(seconds=INTERICTAL_DIST_SEC)
    windows = []
    for b_idx, block in enumerate(blocks):
        for fi in block['files']:
            lo, hi = fi['start'], fi['end']
            excluded = []
            for s in seizures:
                e0, e1 = max(s['onset_abs'] - margin, lo), min(s['offset_abs'] + margin, hi)
                if e0 < e1:
                    excluded.append((e0, e1))
            excluded.sort()
            merged = []
            for e0, e1 in excluded:
                if merged and e0 <= merged[-1][1]:
                    merged[-1] = (merged[-1][0], max(merged[-1][1], e1))
                else:
                    merged.append((e0, e1))
            cursor = lo
            for e0, e1 in merged:
                if cursor < e0:
                    windows.append((cursor, e0, b_idx))
                cursor = max(cursor, e1)
            if cursor < hi:
                windows.append((cursor, hi, b_idx))
    return sorted(windows, key=lambda w: w[0])


def split_interictal_windows(windows, chunk_sec=INTERICTAL_ASSIGNMENT_CHUNK_SEC):
    """Split the global pool into deterministic, non-overlapping chunks."""
    pieces = []
    for start, end, b_idx in windows:
        cursor = start
        while (end - cursor).total_seconds() >= EPOCH_LEN_SEC:
            piece_end = min(end, cursor + timedelta(seconds=chunk_sec))
            pieces.append((cursor, piece_end, b_idx))
            cursor = piece_end
    return pieces


def assign_windows_to_seizures(windows, seizure_ids):
    """Greedily balance disjoint interictal chunks among candidate seizures."""
    ids = sorted(seizure_ids)
    assigned = {sid: [] for sid in ids}
    capacity = {sid: 0 for sid in ids}
    for piece in split_interictal_windows(windows):
        if not ids:
            break
        sid = min(ids, key=lambda x: (capacity[x], x))
        assigned[sid].append(piece)
        capacity[sid] += window_epoch_capacity([piece])
    return assigned



# ==========================================================================
# 5.1 ELIGIBILITY AUDIT
# ==========================================================================
def window_epoch_capacity(windows):
    return int(sum(max(0.0, (end - start).total_seconds()) // EPOCH_LEN_SEC
                   for start, end, _ in windows))


def audit_eligible_seizures(blocks, all_seizures):
    preictal_ok = {}
    for seizure in all_seizures:
        epochs = get_available_preictal_epochs(
            blocks[seizure['block_idx']], seizure, all_seizures)
        preictal_ok[seizure['seizure_id']] = epochs is not None
        if epochs is not None:
            del epochs
    gc.collect()

    candidate_ids = [sid for sid, ok in preictal_ok.items() if ok]
    inter_windows = get_interictal_windows(blocks, all_seizures)
    assigned = assign_windows_to_seizures(inter_windows, candidate_ids)

    rows, eligible_ids = [], []
    for seizure in all_seizures:
        sid = seizure['seizure_id']
        full_preictal = preictal_ok[sid]
        n_interictal = window_epoch_capacity(assigned.get(sid, []))
        adequate_interictal = n_interictal >= MIN_INTERICTAL_EPOCHS_PER_SEIZURE
        eligible = full_preictal and adequate_interictal
        if eligible:
            eligible_ids.append(sid)
        rows.append({
            'SeizureID': sid, 'Onset': seizure['onset_abs'].isoformat(),
            'Offset': seizure['offset_abs'].isoformat(),
            'Full_30min_Available_Preictal': int(full_preictal),
            'Expected_Preictal_Epochs': EXPECTED_PREICTAL_EPOCHS if full_preictal else 0,
            'Allocated_Interictal_EpochCapacity': n_interictal,
            'Adequate_Interictal': int(adequate_interictal), 'Eligible': int(eligible),
            'Reason': ('eligible' if eligible else
                       'fewer_than_180_valid_preictal_epochs' if not full_preictal else
                       'insufficient_allocated_interictal'),
        })
    return rows, eligible_ids


def write_qc_csv(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not rows:
        return
    # Cohort rows may contain an additional Error field only for failed subjects.
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)



# ==========================================================================


In [ ]:
def crop_to_epochs(raw, t_start, t_end, epoch_len=EPOCH_LEN_SEC, expected_epochs=None):
    sample_period = 1.0 / float(raw.info['sfreq'])
    # raw.times[-1] is the time of the final sample, whereas interval endpoints
    # are exclusive. Add one sample so a complete final 10-s epoch is not lost.
    available_end = raw.times[-1] + sample_period
    t_end = min(t_end, available_end)
    if t_end - t_start < epoch_len:
        return None
    cropped = raw.copy().crop(
        tmin=max(t_start, 0), tmax=min(t_end - sample_period, raw.times[-1]),
        include_tmax=True)
    epochs = mne.make_fixed_length_epochs(
        cropped, duration=epoch_len, reject_by_annotation=True, preload=True, verbose=False)
    del cropped
    epochs.drop_bad()
    if len(epochs) == 0:
        del epochs
        gc.collect()
        return None
    if expected_epochs is not None and len(epochs) != expected_epochs:
        print(f'[EXCLUDE WINDOW] expected {expected_epochs} epochs, obtained {len(epochs)}')
        del epochs
        gc.collect()
        return None
    if epochs.get_data(copy=False).shape[1] != len(CHANNELS_18):
        raise AssertionError('epoch data do not contain exactly 18 channels')
    return epochs


def epoch_start_seconds(epochs, reference_raw):
    return (epochs.events[:, 0] - reference_raw.first_samp) / reference_raw.info['sfreq']


def _epoch_overlaps(epoch_start_abs, epoch_end_abs, lo, hi):
    return epoch_start_abs < hi and epoch_end_abs > lo


def get_available_preictal_epochs(block, seizure, all_seizures):
    """Return the latest exactly 180 valid epochs within a 35-min lookback."""
    window = get_preictal_window(seizure, block, all_seizures)
    if window is None:
        return None
    epochs = crop_to_epochs(block['raw'], *window)
    if epochs is None:
        return None
    valid = []
    for idx, sec in enumerate(epoch_start_seconds(epochs, block['raw'])):
        a0 = data_sec_to_abs(block, float(sec))
        a1 = a0 + timedelta(seconds=EPOCH_LEN_SEC)
        if a1 > seizure['onset_abs']:
            continue
        contaminated = False
        for other in all_seizures:
            if other['seizure_id'] != seizure['seizure_id'] and _epoch_overlaps(
                    a0, a1, other['onset_abs'],
                    other['offset_abs'] + timedelta(seconds=POSTICTAL_SEC)):
                contaminated = True
                break
        if not contaminated:
            valid.append(idx)
    if len(valid) < EXPECTED_PREICTAL_EPOCHS:
        del epochs
        gc.collect()
        return None
    selected = epochs[valid[-EXPECTED_PREICTAL_EPOCHS:]]
    del epochs
    return selected


def save_epochs_in_batches(epochs, out_dir, start_batch_idx=0, batch_size=BATCH_SIZE, tag=''):
    n, batch_idx = len(epochs), start_batch_idx
    for i in range(0, n, batch_size):
        print(f'{tag} | Epochs: {n} | Batch: {i}:{min(i + batch_size, n)}')
        batch = epochs[i:i + batch_size].get_data(copy=False)
        save_fold_npy(batch, batch_idx, out_dir)
        batch_idx += 1
        del batch
    return batch_idx


def classify_epoch_phase(epoch_start_abs, epoch_end_abs, target_seizure, all_seizures):
    for s in all_seizures:
        if _epoch_overlaps(epoch_start_abs, epoch_end_abs, s['onset_abs'], s['offset_abs']):
            return 'ictal'
    for s in all_seizures:
        if _epoch_overlaps(epoch_start_abs, epoch_end_abs, s['offset_abs'],
                           s['offset_abs'] + timedelta(seconds=POSTICTAL_SEC)):
            return 'postictal'
    margin = timedelta(seconds=INTERICTAL_DIST_SEC)
    fully_interictal = all(
        epoch_end_abs <= s['onset_abs'] - margin or
        epoch_start_abs >= s['offset_abs'] + margin for s in all_seizures)
    return 'interictal' if fully_interictal else 'excluded'



# ==========================================================================


In [ ]:
# ==========================================================================
# 2. MODELREADY KEYS AND TIMESTAMP-ONLY RECONSTRUCTION
# ==========================================================================
def is_testing_stem(stem):
    return re.search(r'_block_\d+$', stem) is not None


def load_expected_training_metadata(subject):
    paths = sorted((MODEL_ROOT / 'Metadata' / subject).glob('*.parquet'))
    paths = [path for path in paths if not is_testing_stem(path.stem)]
    if not paths:
        raise FileNotFoundError(f'{subject}: no training metadata parquet files')
    frames = [pd.read_parquet(path) for path in paths]
    expected = pd.concat(frames, ignore_index=True)
    required = ['Patient', 'Class', 'SeizureID', 'Epoch']
    missing = [column for column in required if column not in expected.columns]
    if missing:
        raise KeyError(f'{subject}: ModelReady metadata missing {missing}')
    expected = expected[required].copy()
    expected['Patient'] = expected['Patient'].astype(str)
    expected['Class'] = expected['Class'].astype(str).str.lower()
    expected['SeizureID'] = expected['SeizureID'].astype(int)
    expected['Epoch'] = expected['Epoch'].astype(int)
    if expected.duplicated(required).any():
        raise AssertionError(f'{subject}: duplicate ModelReady training keys')
    return expected.sort_values(required).reset_index(drop=True)


def source_edf_for_time(block, when):
    for file_info in block['files']:
        if file_info['start'] <= when < file_info['end']:
            return file_info['file']
    # Exact final sample boundary belongs to the final EDF.
    final_info = block['files'][-1]
    if when == final_info['end']:
        return final_info['file']
    raise ValueError(f'Could not map {when} to an EDF in the selected block')


def preictal_timestamp_rows(subject, blocks, all_seizures, eligible_ids):
    rows = []
    for seizure in all_seizures:
        sid = seizure['seizure_id']
        if sid not in eligible_ids:
            continue
        block = blocks[seizure['block_idx']]
        epochs = get_available_preictal_epochs(block, seizure, all_seizures)
        if epochs is None or len(epochs) != EXPECTED_PREICTAL_EPOCHS:
            raise AssertionError(f'{subject}/seizure {sid}: preictal reconstruction failed')
        starts = epoch_start_seconds(epochs, block['raw'])
        for epoch_idx, start_sec in enumerate(starts):
            start_abs = data_sec_to_abs(block, float(start_sec))
            rows.append({
                'Patient': subject,
                'Class': 'preictal',
                'SeizureID': int(sid),
                'Epoch': int(epoch_idx),
                'EpochStart': start_abs.isoformat(),
                'Block': int(seizure['block_idx']),
                'SourceEDF': source_edf_for_time(block, start_abs),
            })
        del epochs
    gc.collect()
    return rows


def interictal_timestamp_rows(subject, blocks, all_seizures, eligible_ids):
    # This is the timestamp-only equivalent of extract_interictal() in V3.
    windows = get_interictal_windows(blocks, all_seizures)
    windows_by_seizure = assign_windows_to_seizures(windows, eligible_ids)
    rows = []

    for sid in sorted(eligible_ids):
        valid_windows = []
        for abs_start, abs_end, block_idx in windows_by_seizure.get(sid, []):
            block = blocks[block_idx]
            data_start = abs_to_data_sec(block, abs_start)
            data_end = abs_to_data_sec(block, abs_end)
            epochs = crop_to_epochs(block['raw'], data_start, data_end)
            if epochs is not None:
                valid_windows.append(
                    (abs_start, abs_end, block_idx, len(epochs)))
                del epochs

        pool_size = sum(item[3] for item in valid_windows)
        if pool_size < MAX_INTERICTAL_EPOCHS_PER_SEIZURE:
            raise AssertionError(
                f'{subject}/seizure {sid}: only {pool_size}/540 interictal epochs')

        selected_global = np.linspace(
            0, pool_size - 1, MAX_INTERICTAL_EPOCHS_PER_SEIZURE,
            dtype=np.int64)

        cursor = 0
        selected_times = []
        for abs_start, abs_end, block_idx, n_epochs in valid_windows:
            mask = ((selected_global >= cursor)
                    & (selected_global < cursor + n_epochs))
            local_indices = selected_global[mask] - cursor
            cursor += n_epochs
            if len(local_indices) == 0:
                continue

            block = blocks[block_idx]
            data_start = abs_to_data_sec(block, abs_start)
            data_end = abs_to_data_sec(block, abs_end)
            epochs = crop_to_epochs(block['raw'], data_start, data_end)
            if epochs is None or len(epochs) != n_epochs:
                raise RuntimeError(
                    'Interictal epoch count changed between selection passes')
            starts = epoch_start_seconds(epochs, block['raw'])
            for local_idx in local_indices:
                start_abs = data_sec_to_abs(block, float(starts[int(local_idx)]))
                selected_times.append((start_abs, block_idx,
                                       source_edf_for_time(block, start_abs)))
            del epochs

        if len(selected_times) != MAX_INTERICTAL_EPOCHS_PER_SEIZURE:
            raise AssertionError(
                f'{subject}/seizure {sid}: selected {len(selected_times)}/540')
        if selected_times != sorted(selected_times, key=lambda item: item[0]):
            raise AssertionError(
                f'{subject}/seizure {sid}: interictal order is not chronological')

        for epoch_idx, (start_abs, block_idx, source_edf) in enumerate(selected_times):
            rows.append({
                'Patient': subject,
                'Class': 'interictal',
                'SeizureID': int(sid),
                'Epoch': int(epoch_idx),
                'EpochStart': start_abs.isoformat(),
                'Block': int(block_idx),
                'SourceEDF': source_edf,
            })
        gc.collect()
    return rows


In [ ]:
# ==========================================================================
# 3. PROCESS ONE SUBJECT WITH STRICT MODELREADY VALIDATION
# ==========================================================================
KEYS = ['Patient', 'Class', 'SeizureID', 'Epoch']


def release_timeline(blocks):
    for block in blocks:
        try:
            block['raw'].close()
        except Exception:
            pass
        for file_info in block.get('files', []):
            try:
                file_info['raw'].close()
            except Exception:
                pass
    del blocks
    gc.collect()


def process_timestamp_subject(subject):
    expected = load_expected_training_metadata(subject)
    expected_sids = sorted(expected['SeizureID'].unique().tolist())
    subject_dir = RAW_ROOT / subject
    if not subject_dir.exists():
        raise FileNotFoundError(subject_dir)

    print(f'\n===== {subject} =====')
    blocks = build_timeline(str(subject_dir), subject)
    try:
        all_seizures = collect_seizures(blocks)
        audit_rows, eligible_ids = audit_eligible_seizures(blocks, all_seizures)
        eligible_ids = sorted(map(int, eligible_ids))

        if eligible_ids != expected_sids:
            raise AssertionError(
                f'{subject}: segmentation eligible IDs {eligible_ids} '
                f'!= ModelReady IDs {expected_sids}')

        rows = []
        rows.extend(preictal_timestamp_rows(
            subject, blocks, all_seizures, eligible_ids))
        rows.extend(interictal_timestamp_rows(
            subject, blocks, all_seizures, eligible_ids))
        manifest = pd.DataFrame(rows)
        manifest['EpochStart'] = pd.to_datetime(
            manifest['EpochStart'], utc=True)

        if manifest.duplicated(KEYS).any():
            duplicated = manifest.loc[manifest.duplicated(KEYS, keep=False), KEYS]
            raise AssertionError(
                f'{subject}: duplicate timestamp keys\n{duplicated.head()}')

        joined = expected.merge(
            manifest, on=KEYS, how='left', validate='one_to_one', indicator=True)
        matched = int(joined['_merge'].eq('both').sum())
        missing = int(joined['_merge'].ne('both').sum())
        extra = int(len(manifest.merge(
            expected, on=KEYS, how='left', indicator=True)
            .query('_merge != "both"')))

        expected_counts = expected.groupby(['SeizureID', 'Class']).size().unstack(fill_value=0)
        actual_counts = manifest.groupby(['SeizureID', 'Class']).size().unstack(fill_value=0)
        if not expected_counts.equals(actual_counts):
            raise AssertionError(
                f'{subject}: class counts differ\nExpected:\n{expected_counts}'
                f'\nActual:\n{actual_counts}')
        if missing or extra or matched != len(expected):
            raise AssertionError(
                f'{subject}: coverage failure matched={matched}, '
                f'missing={missing}, extra={extra}')

        if manifest['EpochStart'].isna().any():
            raise AssertionError(f'{subject}: missing EpochStart after reconstruction')

        subject_path = OUTPUT_ROOT / 'subjects' / f'{subject}_training_timestamps.parquet'
        manifest.sort_values(KEYS).to_parquet(subject_path, index=False)
        pd.DataFrame(audit_rows).to_csv(
            OUTPUT_ROOT / 'subjects' / f'{subject}_segmentation_audit.csv', index=False)

        qc = {
            'Subject': subject,
            'ExpectedRows': int(len(expected)),
            'ManifestRows': int(len(manifest)),
            'MatchedRows': matched,
            'MissingRows': missing,
            'ExtraRows': extra,
            'EligibleSeizures': len(eligible_ids),
            'Coverage': matched / len(expected),
            'Status': 'PASS',
        }
        pd.DataFrame([qc]).to_csv(
            OUTPUT_ROOT / 'checkpoints' / f'{subject}.csv', index=False)
        print('PASS:', qc)
        return manifest.sort_values(KEYS).reset_index(drop=True), qc
    finally:
        release_timeline(blocks)


In [ ]:
# ==========================================================================
# 4. RUN OR RESUME THE COHORT
# ==========================================================================
for subject in subjects:
    checkpoint = OUTPUT_ROOT / 'checkpoints' / f'{subject}.csv'
    subject_output = OUTPUT_ROOT / 'subjects' / f'{subject}_training_timestamps.parquet'
    if checkpoint.exists() and subject_output.exists():
        existing = pd.read_csv(checkpoint)
        if len(existing) == 1 and existing.loc[0, 'Status'] == 'PASS':
            print('Skip completed:', subject)
            continue
    try:
        process_timestamp_subject(subject)
    except Exception:
        print(f'[FAILED] {subject}')
        traceback.print_exc()
        raise


In [ ]:
# ==========================================================================
# 5. FINAL COHORT QC AND EXPORT
# ==========================================================================
qc_paths = [OUTPUT_ROOT / 'checkpoints' / f'{subject}.csv' for subject in subjects]
missing_qc = [str(path) for path in qc_paths if not path.exists()]
if missing_qc:
    raise AssertionError(f'Missing subject checkpoints: {missing_qc}')

qc = pd.concat([pd.read_csv(path) for path in qc_paths], ignore_index=True)
if len(qc) != len(subjects) or not qc['Status'].eq('PASS').all():
    raise AssertionError('Not every V12 subject passed timestamp reconstruction')
if not np.allclose(qc['Coverage'], 1.0):
    raise AssertionError('Final timestamp coverage is not 100%')

manifest_paths = [
    OUTPUT_ROOT / 'subjects' / f'{subject}_training_timestamps.parquet'
    for subject in subjects
]
manifest = pd.concat(
    [pd.read_parquet(path) for path in manifest_paths], ignore_index=True)

if manifest.duplicated(KEYS).any():
    raise AssertionError('Duplicate keys in final cohort manifest')
manifest['EpochStart'] = pd.to_datetime(manifest['EpochStart'], utc=True)
manifest = manifest.sort_values(KEYS).reset_index(drop=True)

manifest.to_parquet(
    OUTPUT_ROOT / 'training_epoch_timestamps.parquet', index=False)
manifest.to_csv(
    OUTPUT_ROOT / 'training_epoch_timestamps.csv', index=False)
qc.to_csv(OUTPUT_ROOT / 'v12c_qc_summary.csv', index=False)

run_info = {
    'source_segmentation': 'CHB-MIT_Segmentation_GAPAWARE_V3',
    'model_ready_source': str(MODEL_ROOT),
    'raw_source': str(RAW_ROOT),
    'subjects': subjects,
    'rows': int(len(manifest)),
    'seizures': int(manifest[['Patient', 'SeizureID']].drop_duplicates().shape[0]),
    'coverage': 1.0,
    'epoch_length_seconds': EPOCH_LEN_SEC,
    'preictal_minutes': PREICTAL_SEC // 60,
    'interictal_distance_hours': INTERICTAL_DIST_SEC / 3600,
    'postictal_exclusion_hours': POSTICTAL_SEC / 3600,
    'maximum_tolerated_short_gap_seconds': MAX_SHORT_GAP_SEC,
}
with open(OUTPUT_ROOT / 'v12c_run_info.json', 'w', encoding='utf-8') as file:
    json.dump(run_info, file, indent=2)

display(qc)
print('\nFINAL V12C PASS')
print('Rows:', len(manifest))
print('Unique patient-seizure folds:', run_info['seizures'])
print('Coverage: 100%')
print('Output:', OUTPUT_ROOT)


## Using the V12C output in V12B

1. Save the complete `/kaggle/working/MBEC_V12C_TRAINING_TIMESTAMPS`
   folder as a Kaggle dataset.
2. Attach that dataset together with the existing ModelReady and ALL21 output
   to the time-ready V12B notebook.
3. Run `GUARD10` first, then `FORWARD`.

Do not use the timestamp output if the final cell does not print
`FINAL V12C PASS` and `Coverage: 100%`.
